# How clean is Europe's power right now?

Obsyd derives an **estimated hourly CO₂ intensity** for every bidding zone —
the published ENTSO-E generation mix times per-technology emission factors
(IPCC AR5 lifecycle medians via Electricity Maps' open factor table).
`co2.intensity.lifecycle` is the cross-country comparison standard;
`co2.intensity.direct` counts combustion only. Production-based (imports are
not traced), technology-average — an estimate on the public record, not a
measurement. Methodology: `backend/power/co2.py` in the Obsyd repo.


In [ ]:
# pip install obsyd matplotlib
import datetime as dt
import matplotlib.pyplot as plt
from obsyd import Obsyd

ob = Obsyd()
END = dt.date.today()  # the derived series trails the mix by only a few hours
START = END - dt.timedelta(days=14)
ZONES = ['FR', 'DE_LU', 'PL', 'ES']  # nuclear, transition, coal, solar


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for z in ZONES:
    df = ob.series('co2.intensity.lifecycle', z, start=START, end=END)
    ax.plot(df.index, df['value'], lw=1, label=z)
ax.set_ylabel('gCO2eq/kWh (lifecycle, est.)')
ax.legend()
ax.set_title('Two weeks of estimated carbon intensity — four very different grids')
plt.tight_layout()


France barely moves (nuclear baseload), Poland barely moves either (coal —
flat for the opposite reason), while Germany and Spain swing by hundreds of
g/kWh with the sun and wind. The *daily shape* is the story: a solar-heavy
grid is cleanest at noon and dirtiest in the evening ramp.


In [ ]:
# Every zone, latest reading, one request — the live carbon ranking.
wide = ob.snapshot('co2.intensity.lifecycle', hours=12)
latest = wide.ffill().iloc[-1].dropna().sort_values()
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(latest.index, latest.values)
ax.set_xlabel('gCO2eq/kWh (lifecycle, est.) — latest computed hour per zone')
ax.set_title("Europe's zones, cleanest to dirtiest, right now")
plt.tight_layout()


The lifecycle−direct gap is itself a finding — it is the fuel chain
(construction, extraction, methane slip): compare
`ob.series('co2.intensity.direct', 'FR', start=START)` against lifecycle and
nuclear France 'rises' from ~0 to ~12 g/kWh. Estimated, not measured — and
reconcilable: the factor table and every mapping decision are in the open.
